## Model of "RAELLA: Reforming the Arithmetic for Efficient, Low-Resolution, and Low-Loss Analog PIM: No Retraining Required!", ISCA 2023

Paper by Tanner Andrulis, Joel S. Emer, Vivienne Sze

### Description of RAELLA

RAELLA is a ReRAM-based analog CiM accelerator that enables low-energy high-accuracy
analog-domain computation. To enable high accuracy, it uses several slicing and encoding
strategies to enable high-fidelity full-precision arithmetic.

### Tile Level

Eight macros are organized into a tile. Each tile includes a 64kB eDRAM buffer storing
8b inputs/outputs, digital maxpool units, and quantization circuits. RAELLA digitally
computes 8b per-channel quantization, allocating 32b per output channel to store a FP16
quantization scale and bias, or 32kB per tile.


- *Input Path*: Inputs are stored in the eDRAM. An inter-macro network sends inputs to
  macros in the tile.
- *Weight Path*: Weights are kept static in inference and do not move through this
  level.
- *Output Path*: Outputs are gathered from macros via the inter-tile network. They are
  quantized to 8b before being written to the eDRAM.

Next, there are 8 macros in each tile. Inputs and outputs can be spatially reused across
macros with a multicast/reduction network.

### Macro Level

Four arrays are organized into a macro with an input buffer. An input network sends
input vectors to arrays, and if all inputs are shared between two arrays, the input
vector is multicast. To exploit temporal input reuse, the input buffer stores reused
inputs between array cycles. The four arrays can process up to 4× 512 = 2048 inputs
across all rows, so the buffer is sized 2kB. To support Center+Offset weights, each
macro includes a weight center buffer and digital addition circuitry to calculate input
sums. A running sum is kept for each array. To exploit input reuse, we add inputs to the
sum when they are first used in array columns and subtract when they are last used. If
different array columns use different subsets of the inputs, RAELLA adds/subtracts
inputs in a streaming fashion while processing columns.

- *Input Path*: Inputs are stored in the input buffer. The input buffer includes
  circuitry to maintain a running sum of inputs for Center+Offset encoding.
- *Weight Path*: Weights are kept static in inference and do not move through this
  level.
- *Output Path*: Outputs pass through a Center+Offset correction circuit that uses the
  input running sum to calculate the final output.
  
Next, there are 4 arrays in each macro. Inputs and outputs can be spatially reused
across arrays with a multicast/reduction network.

### Array Level

Arrays consist of 512 × 512 2T2Rs. Each array is programmed with weights from one DNN
layer, and each weight filter uses 2-8 array columns based on the DNN layer slicing.
4-bit pulse-train DACs encode inputs, and 7-bit ADCs convert outputs from each column.

7b signed ADC results are summed by shift+add circuits and accumulated in 16b output
buffers. With the most-dense slicing of two slices per weight, one array may produce up
to 256 outputs, which are stored in a 256-entry output buffer. Each entry stores a 16b
output + 8b success flags, for a 768B output buffer total capacity.

RAELLA supports speculation, where it runs the array with a 2-4b input slice, records
which columns saturate, and then reprocesses only the failed columns using 2-4 1b input
slices. RAELLA also supports recovery-only, where it runs 1b input slices only,
processing all columns for these slices. Speculation saves energy by reducing the number
of ADC conversions, but it increases latency due to the added cycles.

In speculation/recovery cycles, inputs are streamed to arrays once for each cycle. In
speculation, ADCs process all columns. If an ADC saturates, the output is not updated
and the success flag is marked. In recovery, all success flags are checked. ADCs process
and write results only for columns that failed speculation. 

If inputs are signed, RAELLA processes positive/negative inputs in two separate cycles.

- *Input Path*: Inputs pass through a 4-bit DACs and appear on the rows of the array.
- *Weight Path*: Weights are stored in the array and are not moved during inference.
- *Output Path*: Outputs are read from the columns of the array with 7-bit ADCs. They
  are shifted+added before being stored in a psum register. The psum register also
  stores speculation success flags.

Next, there are 512 columns in each array. Inputs are reused between columns (*i.e.,*
each input-carrying wire connects to all columns), while outputs and weights are not
reused.

### Column Level

Each column consists of 512 2T2Rs ReRAM devices. Depending on how RAELLA slices weights,
columns may store 1b-4b weight slices (2-8 slices per weight value).

- *Input Path*: Each input is passed directly to a row in the column.
- *Weight Path*: Weights are not moved during inference.
- *Output Path*: Outputs pass through a current mirror to buffer their values before
  exiting the column.
  
### Row Level

Each row in a column has one 2T2R device which stores a differentially-encoded weight
value.

- *Input Path*: The input is used for a MAC operation.
- *Weight Path*: A 2-4b weight is stored in the 2T2R device and is used for a MAC
  operation.
- *Output Path*: The output is supplied by a MAC operation.

In [ ]:
from _scripts import *
from tqdm.auto import tqdm

display_important_variables("raella_isca_2023")
get_spec("raella_isca_2023").arch

### Energy, Area, and Latency Breakdown: Speculation On vs. Off

This test explores the energy, area, and latency of the accelerator computing MVM
operations with and without speculation.

We find that speculation increases array and DAC energy due to additional speculation
cycles, and increases the input buffer energy due to 2× fetches of inputs (for
speculation & recovery cycles). It decreases output buffer energy and ADC energy due to
fewer analog outputs being read from the array and written to the output buffer.

In [ ]:
def _run_raella(speculation_enabled):
    spec = get_spec("raella_isca_2023", add_dummy_main_memory=True)
    spec.arch.variables["speculation_enabled"] = int(speculation_enabled)
    result = spec.map_workload_to_arch(print_progress=False)
    ev = spec.calculate_component_costs()
    area = {k: float(v) for k, v in ev.arch.per_component_total_area.items()}
    return speculation_enabled, result, area


runs = parallel([delayed(_run_raella)(b) for b in [False, True]])
results = {f"Speculation {b}": r for b, r, _ in runs}
areas = {f"Speculation {b}": a for b, _, a in runs}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

energy_fj = {
    label: {
        k: v * 1e15
        for k, v in r.per_compute().energy(per_component=True).items()
        if v > 0
    }
    for label, r in results.items()
}
bar_stacked(energy_fj, "Configuration", "Energy (fJ/MAC)", "Energy Breakdown", axes[0])

area_um2 = {
    label: {k: v * 1e12 for k, v in a.items() if v > 0} for label, a in areas.items()
}
bar_stacked(area_um2, "Configuration", "Area (um^2)", "Area Breakdown", axes[1])

latency_us = {label: {"": r.latency() * 1e6} for label, r in results.items()}
bar_stacked(
    latency_us,
    "Configuration",
    "Latency (us)",
    "Matrix-Vector Multiplication Latency Breakdown",
    axes[2],
)

plt.tight_layout()
plt.show()

for label, r in results.items():
    tops = 2 / r.per_compute().latency() / 1e12
    tops_w = 2 / r.per_compute().energy() / 1e12
    print(
        f"{label}: TOPS={tops:.4f}, TOPS/W={tops_w:.1f}, latency={r.latency() * 1e6:.2f} us"
    )

In [ ]:
results["Speculation True"].evaluated_specs["Matmul"].arch[
    "CimUnit"
].component_modeling_log

In [ ]:
results["Speculation True"].latency(per_component=True)

### Running Full DNNs on the Accelerator

This test explores the energy, area, and latency of the accelerator when running full
DNN workloads with and without speculation.

We can observe the following:

- Speculation generally reduces energy and increases latency due to the factors
  described in the previous test.
- RAELLA is more energy efficient for layers with more input channels due to higher
  utilization of arrays (which leads to lower ADC energy, per the Titanium Law described
  in the paper).
- Energy and latency can also increase for layers with signed inputs (e.g., many
  Transformer layers) because RAELLA processes signed inputs in two different cycles,
  which increases the number of array activations and ADC uses.


In [ ]:
MAX_EINSUMS = 10


def _run_raella_dnn(dnn, speculation):
    workload_path = getattr(af.examples.workloads.compute_in_memory, dnn)
    parse = {"BATCH_SIZE": 1}
    workload = af.Workload.from_yaml(
        workload_path, top_key="workload", jinja_parse_data=parse
    )
    renames = af.Renames.from_yaml(
        workload_path, top_key="renames", jinja_parse_data=parse
    )

    spec = get_spec("raella_isca_2023", add_dummy_main_memory=True, n_macros=1)
    spec.arch.variables["speculation_enabled"] = int(speculation)
    workload.einsums = workload.einsums[:MAX_EINSUMS]
    spec.workload = workload
    spec.renames = renames

    return spec.map_workload_to_arch(
        one_pbar_only=True
    ).drop_components_with_zero_energy_and_latency()


dnns = ["resnet18", "mobilenet_v3", "gpt2_medium"]

In [ ]:
pairs = [(d, s) for d in dnns for s in [False, True]]
dnn_results = {
    (d, s): _run_raella_dnn(d, s) for d, s in tqdm(pairs, desc="DNN x speculation")
}

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

energy_fj = {
    f"{d} Speculation {s}": {
        k: v * 1e15
        for k, v in dnn_results[(d, s)].per_compute().energy(per_component=True).items()
        if v > 0
    }
    for d in dnns
    for s in [False, True]
}
bar_stacked(energy_fj, "DNN", "Energy (fJ/MAC)", "Per-MAC Energy", axes[0])

total_energy_mj = {
    f"{d} Speculation {s}": {
        k: v * 1e3
        for k, v in dnn_results[(d, s)].energy(per_component=True).items()
        if v > 0
    }
    for d in dnns
    for s in [False, True]
}
bar_stacked(total_energy_mj, "DNN", "Energy (mJ)", "Total Energy", axes[1])

tops = {d: {} for d in dnns}
tops_per_w = {d: {} for d in dnns}
for d in dnns:
    for s in [False, True]:
        r = dnn_results[(d, s)]
        tops[d][f"Speculation {s}"] = 2 / r.per_compute().latency() / 1e12
        tops_per_w[d][f"Speculation {s}"] = 2 / r.per_compute().energy() / 1e12
bar_comparison(tops, "DNN", "TOPS", "Full-DNN Throughput", axes[2])
bar_comparison(tops_per_w, "DNN", "TOPS/W", "Full-DNN Energy Efficiency", axes[3])

plt.tight_layout()
plt.show()

print(
    f"{'DNN':<14} {'Spec':<6} {'TOPS':>8} {'TOPS/W':>10} {'Latency(us)':>12} {'Energy(mJ)':>12}"
)
for d in dnns:
    for s in [False, True]:
        r = dnn_results[(d, s)]
        print(
            f"{d:<14} {str(s):<6} "
            f"{2 / r.per_compute().latency() / 1e12:8.4f} "
            f"{2 / r.per_compute().energy() / 1e12:10.2f} "
            f"{r.latency() * 1e6:12.2f} {r.energy() * 1e3:12.4f}"
        )

### Per-Layer Results

For each DNN we plot four panels:

1. Per-layer per-component energy with speculation enabled.
2. Per-layer per-component energy with speculation disabled.
3. Per-layer latency comparing speculation on vs off.
4. Per-layer total energy comparing speculation on vs off.

RAELLA is more energy efficient for layers with more input channels because of
higher utilization of arrays (lower ADC energy per the Titanium Law in the
paper). Energy and latency can also increase for layers with signed inputs
(e.g., many Transformer layers) because RAELLA processes signed inputs in two
separate cycles.

In [ ]:
for d in dnns:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for ax, speculation_enabled in zip(axes[:2], [True, False]):
        r = dnn_results[(d, speculation_enabled)]
        einsum_names = list(r.einsum_names)
        per_layer_e = {}
        for i, einsum in enumerate(einsum_names):
            er = r.per_compute(per_einsum=True).access(
                einsum, col_idx=0, keep_key_index=True
            )
            per_layer_e[i] = {
                k: v * 1e15 for k, v in er.energy(per_component=True).items() if v > 0
            }
        bar_stacked(
            per_layer_e,
            "Layer",
            "Energy (fJ/MAC)",
            f"{d} Per-Layer Energy Speculation {speculation_enabled}",
            ax,
        )

    layer_names = list(dnn_results[(d, False)].einsum_names)
    for ax, attr, scale, title, ylabel in [
        (axes[2], "latency", 1e6, "Latency", "us"),
        (axes[3], "energy", 1e3, "Total Energy", "mJ"),
    ]:
        per_layer_data = {f"Speculation {s}": {} for s in [False, True]}
        for i, layer in enumerate(layer_names):
            for s in [False, True]:
                vals = getattr(dnn_results[(d, s)], attr)(per_einsum=True)
                per_layer_data[f"Speculation {s}"][i] = float(vals[layer]) * scale
        bar_comparison(
            per_layer_data, "Layer", f"{title} ({ylabel})", f"{d} Per-Layer {title}", ax
        )

    plt.tight_layout()
    plt.show()

### Interactive Results Browser

Pick a DNN and whether speculation is enabled to inspect that run's full
`display_dnn_results` summary (energy/latency/action breakdown plus a per-layer slider).

In [ ]:
from ipywidgets import interact, Dropdown


# Browse the full-DNN results from the experiment above: choose a DNN and a speculation
# setting, and display_dnn_results renders that run's summary, breakdown, and per-layer slider.
@interact(
    dnn=Dropdown(options=dnns, description="DNN"),
    speculation=Dropdown(
        options=[("Off", False), ("On", True)], description="Speculation"
    ),
)
def _show_raella_dnn_results(dnn=dnns[0], speculation=False):
    display_dnn_results(
        dnn_results[(dnn, speculation)],
        title=f"{dnn} (Speculation {'On' if speculation else 'Off'})",
    )